<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/260243025023_Jatin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Try the following algorithms and examine which of these is best fit for precision and recall
- SVM-Radial(tune with few parameter set)
- Decision tree (tune with few parameter set)
- Gaussian NB
- KNN Classifier(tune with few parameter set)

In [16]:
import pandas as pd
city = pd.read_csv("/content/City_Types.csv")

In [17]:
city.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52704 entries, 0 to 52703
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   CO      52704 non-null  int64  
 1   NO2     52704 non-null  float64
 2   SO2     52704 non-null  float64
 3   O3      52704 non-null  int64  
 4   PM2.5   52704 non-null  float64
 5   PM10    52704 non-null  float64
 6   Type    52704 non-null  object 
dtypes: float64(4), int64(2), object(1)
memory usage: 2.8+ MB


In [18]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_score, recall_score

# Classifier Imports
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

In [19]:
city.isna().sum()

,0
CO,0
NO2,0
SO2,0
O3,0
PM2.5,0
PM10,0
Type,0


In [20]:
city.columns

Index(['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'Type'], dtype='object')

In [23]:
city['Type'] = city['Type'].map({'Industrial': 0, 'Residential': 1})

In [24]:
city['Type'].value_counts()

,count
Type,
0,26352
1,26352


In [25]:
X = city.drop('Type', axis=1)
y = city['Type']

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=25023)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((42163, 6), (10541, 6), (42163,), (10541,))

In [27]:
# Distance-based models (SVM, KNN) absolutely require scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = {}

### 2. MODEL TRAINING & TUNING

- Algorithm A: SVM with Radial Basis Function (RBF) + Tuning

In [32]:
svm_param_grid = {
    'C': [ 1, 10],
    'gamma': [0.01, 0.1]
}

In [33]:
svm_grid = GridSearchCV(SVC(kernel='rbf'), svm_param_grid, cv=5, scoring='f1')
svm_grid.fit(X_train_scaled, y_train)
best_svm = svm_grid.best_estimator_
results['SVM (Radial)'] = best_svm.predict(X_test_scaled)
print(f"Best SVM Parameters: {svm_grid.best_params_}")

Best SVM Parameters: {'C': 10, 'gamma': 0.1}


- Algorithm B: Decision Tree

In [34]:
dt = DecisionTreeClassifier(random_state=25023)
dt.fit(X_train, y_train)
results['Decision Tree'] = dt.predict(X_test)

- Algorithm C: Gaussian Naive Bayes

In [35]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)
results['Gaussian NB'] = gnb.predict(X_test)

- Algorithm D: KNN Classifier

In [36]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
results['KNN Classifier'] = knn.predict(X_test_scaled)

### EVALUATION

In [37]:
print("\n=== PERFORMANCE COMPARISON ===")
print(f"{'Algorithm':<20} | {'Precision':<10} | {'Recall':<10}")
print("-" * 48)
for name, y_pred in results.items():
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='binary')
    print(f"{name:<20} | {precision:<10.4f} | {recall:<10.4f}")


=== PERFORMANCE COMPARISON ===
Algorithm            | Precision  | Recall    
------------------------------------------------
SVM (Radial)         | 0.9775     | 0.9881    
Decision Tree        | 0.9816     | 0.9810    
Gaussian NB          | 0.8977     | 0.9537    
KNN Classifier       | 0.9811     | 0.9920    


## 2. Consider the dataset named `student_habits_performance.csv`
- a. Perform Label encoding on column named `diet_quality` using following order poor < fair < good
- b. Apply Square root transform on column `attendance_percentage`
- c. Perform one hot encoding on `gender` column
- d. Apply StandardScalar on `age` column

In [65]:
student = pd.read_csv("/content/student_habits_performance.csv")

- a. Ordinal/Label encoding on diet_quality
- Using map to strictly enforce the poor < fair < good order

In [66]:
diet_mapping = {"poor": 0, "fair": 1, "good": 2}
student["diet_quality"] = student["diet_quality"].map(diet_mapping)
student["diet_quality"]

,diet_quality
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
995,NaN
996,NaN
997,NaN
998,NaN


- b. Apply Square root transform on column attendance_percentage

In [67]:
import numpy as np
student["attendance_percentage"] = np.sqrt(student["attendance_percentage"])
student["attendance_percentage"]

,attendance_percentage
0,9.219544
1,9.864076
2,9.736529
3,8.426150
4,9.534149
...,...
995,8.774964
996,9.273618
997,7.867655
998,10.000000


- c. Perform one hot encoding on gender column

In [68]:
print(student.columns)

Index(['student_id', 'age', 'gender', 'study_hours_per_day',
       'social_media_hours', 'netflix_hours', 'part_time_job',
       'attendance_percentage', 'sleep_hours', 'diet_quality',
       'exercise_frequency', 'parental_education_level', 'internet_quality',
       'mental_health_rating', 'extracurricular_participation', 'exam_score'],
      dtype='object')


In [69]:
student['gender'].value_counts()

,count
gender,
Female,481
Male,477
Other,42


In [70]:
student = pd.get_dummies(student, columns=["gender"], drop_first=False)

In [71]:
student.columns

Index(['student_id', 'age', 'study_hours_per_day', 'social_media_hours',
       'netflix_hours', 'part_time_job', 'attendance_percentage',
       'sleep_hours', 'diet_quality', 'exercise_frequency',
       'parental_education_level', 'internet_quality', 'mental_health_rating',
       'extracurricular_participation', 'exam_score', 'gender_Female',
       'gender_Male', 'gender_Other'],
      dtype='object')

- d. Apply StandardScalar on age column

In [72]:
scaler = StandardScaler()
student["age"] = scaler.fit_transform(student[["age"]])
student["age"]

,age
0,1.084551
1,-0.215870
2,0.217604
3,1.084551
4,-0.649344
...,...
995,0.217604
996,-1.516291
997,-0.215870
998,1.518025


## 3. Consider the dataset `HepatitisCdata.csv`
- Target Column: Category
- a. Do OHE on string columns and fill `NA` with median of respective column
- b. Create simple NN model for classification with 3 hidden layers
- c. Create simple NN model for classification with 2 hidden layers
- d. Compare the models based on precision, recall and log loss. Then explainin your words which is better and why?

In [73]:
hepatitis= pd.read_csv("/content/HepatitisCdata.csv")

In [74]:
hepatitis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 615 entries, 0 to 614
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  615 non-null    int64  
 1   Category    615 non-null    int64  
 2   Age         615 non-null    int64  
 3   Sex         615 non-null    object 
 4   ALB         614 non-null    float64
 5   ALP         597 non-null    float64
 6   ALT         614 non-null    float64
 7   AST         615 non-null    float64
 8   BIL         615 non-null    float64
 9   CHE         615 non-null    float64
 10  CHOL        605 non-null    float64
 11  CREA        615 non-null    float64
 12  GGT         615 non-null    float64
 13  PROT        614 non-null    float64
dtypes: float64(10), int64(3), object(1)
memory usage: 67.4+ KB


In [80]:
hepatitis.shape

(615, 13)

In [75]:
hepatitis.isna().sum()

,0
Unnamed: 0,0
Category,0
Age,0
Sex,0
ALB,1
ALP,18
ALT,1
AST,0
BIL,0
CHE,0


In [76]:
hepatitis.head()

,Unnamed: 0,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT
0,1,0,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0
1,2,0,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5
2,3,0,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3
3,4,0,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7
4,5,0,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7


In [77]:
hepatitis.drop(columns=['Unnamed: 0'],inplace=True)

In [78]:
hepatitis.columns

Index(['Category', 'Age', 'Sex', 'ALB', 'ALP', 'ALT', 'AST', 'BIL', 'CHE',
       'CHOL', 'CREA', 'GGT', 'PROT'],
      dtype='object')

In [88]:
hepatitis['Category'].value_counts()

,count
Category,
0,533
1,82


- a. Do OHE on columns and fill NA with median of respective column

In [83]:
hepatitis_ohe = pd.get_dummies(hepatitis)
hepatitis_ohe.shape

(615, 14)

In [84]:
hepatitis_ohe.isna().sum()

,0
Category,0
Age,0
ALB,1
ALP,18
ALT,1
AST,0
BIL,0
CHE,0
CHOL,10
CREA,0


In [85]:
hepatitis_ohe.fillna(hepatitis_ohe.median(), inplace=True)

In [86]:
hepatitis_ohe.isna().sum().sum()

np.int64(0)

In [87]:
hepatitis_ohe.head()

,Category,Age,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT,Sex_f,Sex_m
0,0,32,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0,False,True
1,0,32,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5,False,True
2,0,32,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3,False,True
3,0,32,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7,False,True
4,0,32,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7,False,True


In [89]:
# Target Column: Category
X = hepatitis_ohe.drop('Category', axis=1)
y = hepatitis_ohe['Category']

- b. Create simple NN model for classification with 3 hidden layers

In [90]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [92]:
# 1. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25023, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((430, 13), (185, 13), (430,), (185,))

In [93]:
# 2. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [94]:
# 3. Build the Neural Network with 3 Hidden Layers
model_3h = Sequential([
    # Input layer and 1st Hidden Layer
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),Dropout(0.2),

    Dense(32, activation='relu'),Dropout(0.2),

    Dense(16, activation='relu'),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [95]:
# 4. Compile the model
model_3h.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [96]:
# 5. Train the model
history_3h = model_3h.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - accuracy: 0.8605 - loss: 0.6411 - val_accuracy: 0.8837 - val_loss: 0.5900
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8708 - loss: 0.5774 - val_accuracy: 0.8837 - val_loss: 0.5330
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8708 - loss: 0.5164 - val_accuracy: 0.8837 - val_loss: 0.4550
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8786 - loss: 0.4589 - val_accuracy: 0.8837 - val_loss: 0.3776
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8837 - loss: 0.3828 - val_accuracy: 0.9070 - val_loss: 0.3072
Epoch 6/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9096 - loss: 0.3316 - val_accuracy: 0.9070 - val_loss: 0.2443
Epoch 7/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.8992 - loss: 0.2978 - val_accuracy: 0.9302 - val_loss: 0.2011
Epoch 8/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9276 - loss: 0.2553 - val_accuracy: 0.9302 - v

- c. Create simple NN model for classification with 2 hidden layers

In [98]:
# 1. Build the Neural Network with 2 Hidden Layers
model_2h = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

In [99]:
# 2. Compile the model
model_2h.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [100]:
# 3. Train the model
history_2h = model_2h.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - accuracy: 0.3618 - loss: 0.7749 - val_accuracy: 0.8605 - val_loss: 0.5861
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7726 - loss: 0.5850 - val_accuracy: 0.9302 - val_loss: 0.4411
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8941 - loss: 0.4814 - val_accuracy: 0.9302 - val_loss: 0.3503
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9121 - loss: 0.3927 - val_accuracy: 0.9302 - val_loss: 0.2755
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9070 - loss: 0.3465 - val_accuracy: 0.9535 - val_loss: 0.2210
Epoch 6/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9251 - loss: 0.2905 - val_accuracy: 0.9535 - val_loss: 0.1818
Epoch 7/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9225 - loss: 0.2570 - val_accuracy: 0.9535 - val_loss: 0.1534
Epoch 8/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9406 - loss: 0.2233 - val_accuracy: 0.9535 - v

- d. Compare the models based on precision, recall and log loss

In [101]:
from sklearn.metrics import precision_score, recall_score, log_loss

y_pred_prob_3h = model_3h.predict(X_test_scaled).ravel()
y_pred_class_3h = (y_pred_prob_3h > 0.5).astype(int)

precision_3h = precision_score(y_test, y_pred_class_3h)
recall_3h = recall_score(y_test, y_pred_class_3h)
loss_3h = log_loss(y_test, y_pred_prob_3h)

y_pred_prob_2h = model_2h.predict(X_test_scaled).ravel()
y_pred_class_2h = (y_pred_prob_2h > 0.5).astype(int)

precision_2h = precision_score(y_test, y_pred_class_2h)
recall_2h = recall_score(y_test, y_pred_class_2h)
loss_2h = log_loss(y_test, y_pred_prob_2h)

print("=== Model Comparison ===")
print(f"3 Hidden Layers -> Precision: {precision_3h:.4f} | Recall: {recall_3h:.4f} | Log Loss: {loss_3h:.4f}")
print(f"2 Hidden Layers -> Precision: {precision_2h:.4f} | Recall: {recall_2h:.4f} | Log Loss: {loss_2h:.4f}")

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
=== Model Comparison ===
3 Hidden Layers -> Precision: 0.9524 | Recall: 0.8000 | Log Loss: 0.1417
2 Hidden Layers -> Precision: 0.9500 | Recall: 0.7600 | Log Loss: 0.1706


- explain in your words which is better and why?

* `3 Hidden Layer model` is the better Because:
  - Significantly Lower Log Loss (0.1417 vs. 0.1706)
  - Crucially Higher Recall (0.8000 vs. 0.7600)
  - Slightly Higher Precision (0.9524 vs. 0.9500)

  - The 3 Hidden Layer model generalized the underlying patterns of this data better, achieving higher clinical safety (Recall) and much better statistical confidence (Log Loss).
